In [2]:
%pip install mysql-connector-python boto3 pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="customerfeedback.c5i420e4ghsy.ap-south-1.rds.amazonaws.com",
    port=3306,
    user="admin",
    password="****",
    database="customerfeedback"
)

customers_df = pd.read_sql("SELECT * FROM customers", conn)
products_df = pd.read_sql("SELECT * FROM products", conn)
feedback_df = pd.read_sql("SELECT * FROM feedback", conn)
sales_df = pd.read_sql("SELECT * FROM sales", conn)

conn.close()

print("Feedback data:")
print(feedback_df)

C:\Users\Saravanan\AppData\Local\Temp\ipykernel_9928\88369698.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers_df = pd.read_sql("SELECT * FROM customers", conn)
C:\Users\Saravanan\AppData\Local\Temp\ipykernel_9928\88369698.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products_df = pd.read_sql("SELECT * FROM products", conn)
C:\Users\Saravanan\AppData\Local\Temp\ipykernel_9928\88369698.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  feedback_df = pd.read_sql("SELECT * FROM feedback", conn)


Feedback data:
   feedback_id  customer_id  product_id  rating  \
0            1            1           1       5   
1            2            2           2       2   
2            3            3           3       4   
3            4            1           4       3   
4            5            2           1       5   

                            comment feedback_date  
0  Excellent mouse, very responsive    2024-01-10  
1         Chair broke within a week    2024-01-12  
2                               NaN    2024-01-15  
3                Average brightness          None  
4             Great value for money    2024-01-18  


C:\Users\Saravanan\AppData\Local\Temp\ipykernel_9928\88369698.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_df = pd.read_sql("SELECT * FROM sales", conn)


In [4]:
# Check for missing values before cleaning
print("Missing values in feedback_df:")
print(feedback_df.isnull().sum())

print("\nMissing values in sales_df:")
print(sales_df.isnull().sum())

Missing values in feedback_df:
feedback_id      0
customer_id      0
product_id       0
rating           0
comment          1
feedback_date    1
dtype: int64

Missing values in sales_df:
sale_id        0
customer_id    0
product_id     0
quantity       0
sale_date      0
dtype: int64


In [5]:
# Handle missing comment - fill with a placeholder instead of leaving blank
feedback_df['comment'] = feedback_df['comment'].fillna('No comment provided')

# Handle missing feedback_date - in real pipelines you'd either:
# (a) drop the row if date is critical for analysis, or
# (b) fill with a default/estimated date
# Here we'll drop it, since a missing date makes trend analysis unreliable
feedback_df = feedback_df.dropna(subset=['feedback_date'])

# Standardize data types
feedback_df['feedback_date'] = pd.to_datetime(feedback_df['feedback_date'])
feedback_df['rating'] = feedback_df['rating'].astype(int)

# Check for duplicate records
duplicates = feedback_df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

# Remove duplicates if any exist
feedback_df = feedback_df.drop_duplicates()

# Confirm no missing values remain
print("\nMissing values after cleaning:")
print(feedback_df.isnull().sum())

print("\nCleaned feedback data:")
print(feedback_df)

Duplicate rows found: 0

Missing values after cleaning:
feedback_id      0
customer_id      0
product_id       0
rating           0
comment          0
feedback_date    0
dtype: int64

Cleaned feedback data:
   feedback_id  customer_id  product_id  rating  \
0            1            1           1       5   
1            2            2           2       2   
2            3            3           3       4   
4            5            2           1       5   

                            comment feedback_date  
0  Excellent mouse, very responsive    2024-01-10  
1         Chair broke within a week    2024-01-12  
2               No comment provided    2024-01-15  
4             Great value for money    2024-01-18  


In [6]:
# Validate sales_df
print("Sales data before curation:")
print(sales_df)

# Standardize data types
sales_df['sale_date'] = pd.to_datetime(sales_df['sale_date'])
sales_df['quantity'] = sales_df['quantity'].astype(int)

# Check for duplicates
dup_sales = sales_df.duplicated().sum()
print(f"\nDuplicate rows in sales: {dup_sales}")
sales_df = sales_df.drop_duplicates()

# Sanity check: quantity should never be zero or negative
invalid_qty = sales_df[sales_df['quantity'] <= 0]
print(f"\nInvalid quantity rows: {len(invalid_qty)}")

print("\nCurated sales data:")
print(sales_df)

Sales data before curation:
   sale_id  customer_id  product_id  quantity   sale_date
0        1            1           1         2  2024-01-05
1        2            2           2         1  2024-01-06
2        3            3           3         5  2024-01-07
3        4            1           4         1  2024-01-08
4        5            2           1         3  2024-01-09

Duplicate rows in sales: 0

Invalid quantity rows: 0

Curated sales data:
   sale_id  customer_id  product_id  quantity  sale_date
0        1            1           1         2 2024-01-05
1        2            2           2         1 2024-01-06
2        3            3           3         5 2024-01-07
3        4            1           4         1 2024-01-08
4        5            2           1         3 2024-01-09


In [7]:
import os

# Create a folder to hold curated files (keeps things organized)
output_folder = "curated_data"
os.makedirs(output_folder, exist_ok=True)

# Save curated dataframes as CSV
feedback_df.to_csv(f"{output_folder}/feedback_curated.csv", index=False)
sales_df.to_csv(f"{output_folder}/sales_curated.csv", index=False)
customers_df.to_csv(f"{output_folder}/customers_curated.csv", index=False)
products_df.to_csv(f"{output_folder}/products_curated.csv", index=False)

print("Files saved successfully in:", os.path.abspath(output_folder))

# List the files to confirm
print("\nFiles created:")
for file in os.listdir(output_folder):
    print(f" - {file}")

Files saved successfully in: C:\Users\Saravanan\curated_data

Files created:
 - customers_curated.csv
 - feedback_curated.csv
 - products_curated.csv
 - sales_curated.csv


In [ ]:
import boto3

# Fill these in yourself - do not share these anywhere
aws_access_key_id = "YOUR_ACCESS_KEY_ID"
aws_secret_access_key = "YOUR_SECRET_ACCESS_KEY"
bucket_name = "sophia-customerfeedback-etl"  # e.g. sophia-customerfeedback-etl-2026
region = "ap-south-1"

s3 = boto3.client(
    's3',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region
)

# Files to upload
files_to_upload = [
    "curated_data/feedback_curated.csv",
    "curated_data/sales_curated.csv",
    "curated_data/customers_curated.csv",
    "curated_data/products_curated.csv"
]

for file_path in files_to_upload:
    file_name = file_path.split("/")[-1]
    s3_key = f"curated/{file_name}"  # stores under a "curated" folder inside the bucket
    s3.upload_file(file_path, bucket_name, s3_key)
    print(f"Uploaded {file_name} to s3://{bucket_name}/{s3_key}")

Uploaded feedback_curated.csv to s3://sophia-customerfeedback-etl/curated/feedback_curated.csv
Uploaded sales_curated.csv to s3://sophia-customerfeedback-etl/curated/sales_curated.csv
Uploaded customers_curated.csv to s3://sophia-customerfeedback-etl/curated/customers_curated.csv
Uploaded products_curated.csv to s3://sophia-customerfeedback-etl/curated/products_curated.csv


In [2]:
%pip install psycopg2-binary sqlalchemy

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 23.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import boto3
import pandas as pd
from sqlalchemy import create_engine

# Reuse your existing S3 client setup
aws_access_key_id = "YOUR_ACCESS_KEY_ID"
aws_secret_access_key = "YOUR_SECRET_ACCESS_KEY"
bucket_name = "sophia-customerfeedback-etl"
region = "ap-south-1"

s3 = boto3.client(
    's3',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region
)

# Download curated files from S3 back into local files (simulates a real "load" step)
files = ["customers_curated.csv", "products_curated.csv", "feedback_curated.csv", "sales_curated.csv"]

for file in files:
    s3.download_file(bucket_name, f"curated/{file}", f"downloaded_{file}")
    print(f"Downloaded {file}")

# Load each into a dataframe
customers_load = pd.read_csv("downloaded_customers_curated.csv")
products_load = pd.read_csv("downloaded_products_curated.csv")
feedback_load = pd.read_csv("downloaded_feedback_curated.csv")
sales_load = pd.read_csv("downloaded_sales_curated.csv")

print("\nAll files loaded into dataframes, ready for Postgres.")

Downloaded customers_curated.csv
Downloaded products_curated.csv
Downloaded feedback_curated.csv
Downloaded sales_curated.csv

All files loaded into dataframes, ready for Postgres.


In [ ]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Postgres connection details
pg_user = "postgres"
pg_password = quote_plus("****")
pg_host = "localhost"
pg_port = "5432"
pg_database = "curateddb"

engine = create_engine(f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_database}")

# Load each dataframe into its matching Postgres table
customers_load.to_sql("customers", engine, if_exists="append", index=False)
products_load.to_sql("products", engine, if_exists="append", index=False)
feedback_load.to_sql("feedback", engine, if_exists="append", index=False)
sales_load.to_sql("sales", engine, if_exists="append", index=False)

print("All curated data loaded into Postgres successfully.")

All curated data loaded into Postgres successfully.
